# Parent-Selection Benchmark Results

This notebook compares DC-first MAP-Elites parent-selection runs across modes and shared random seeds. Run the cells from top to bottom. Leave `RESULT_DIRECTORY` as `None` to analyze the newest result under `data/complex_grid/results/parent_selection`; set it to a specific result directory to revisit an older study. The ranking endpoint is the configured fitness, while N-1 overload energy remains a reported operational guardrail.

In [ ]:
from pathlib import Path

# Set to a result directory to override automatic latest-result discovery.
RESULT_DIRECTORY: Path | None = None
REPOSITORY_ROOT = Path.cwd()
while not (REPOSITORY_ROOT / "pyproject.toml").is_file() and REPOSITORY_ROOT.parent != REPOSITORY_ROOT:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Could not locate the repository root containing pyproject.toml.")
RESULTS_ROOT = REPOSITORY_ROOT / "data/complex_grid/results/parent_selection"

# Analysis contract. Fitness is the ranking endpoint; overload remains a guardrail.
BASELINE_MODE = "UNIi"
FITNESS_COLUMN = "quality.best_fitness"
INITIAL_FITNESS_COLUMN = "quality.initial_fitness"
OVERLOAD_METRIC = "overload_energy_n_1"
ESSENTIAL_METRICS = {
    "overload": "quality.best_candidate_metrics.overload_energy_n_1",
    "severity": "max_flow_n_1",
    "criticality": "critical_branch_count_n_1",
    "operational_cost": ("switching_distance", "split_subs"),
    "diversity": "archive.cell_coverage",
    "efficiency": "execution.cumulative.branch_combinations",
}
BOOTSTRAP_RESAMPLES = 10_000

# Display and export options.
EXPORT_ARTIFACTS = False
EXPORT_MARKDOWN_REPORT = True
SHOW_SEED_TRACES = True
FIGURE_WIDTH = 980
FIGURE_HEIGHT = 460
PALETTE = {
    "UNIi": "#087e8b",
    "UNIc": "#ff5a5f",
    "UCBc": "#6c5ce7",
    "UCBb": "#457b9d",
    "UCBs": "#7a9e9f",
    "Ec": "#f4a261",
    "Xc": "#2a9d8f",
    "G": "#264653",
}

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Result override: {RESULT_DIRECTORY or 'latest available'}")
print(f"Baseline mode: {BASELINE_MODE}; ranking endpoint: configured fitness")

In [ ]:
import json
import warnings

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from toop_engine_topology_optimizer.benchmark.repertoire_analysis import (
    aggregate_seed_projections,
    descriptor_pairs,
    project_repertoire_snapshot,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4g}")

PLOT_TEMPLATE = "plotly_white"
px.defaults.template = PLOT_TEMPLATE
px.defaults.width = FIGURE_WIDTH
px.defaults.height = FIGURE_HEIGHT
px.defaults.color_discrete_map = PALETTE


def metric_column(metric: str) -> str:
    """Return the flattened trajectory column for a best-candidate metric."""
    return f"quality.best_candidate_metrics.{metric}"


def transparent_color(hex_color: str, opacity: float = 0.2) -> str:
    """Convert a six-digit hex color to a Plotly-compatible RGBA color."""
    red, green, blue = (int(hex_color[index : index + 2], 16) for index in (1, 3, 5))
    return f"rgba({red}, {green}, {blue}, {opacity})"

## Resolve Result Directory

The default resolver only considers directories containing `study_manifest.json`, so incomplete timestamp directories are skipped.

In [ ]:
def resolve_result_directory(override: Path | None, results_root: Path) -> Path:
    """Return a requested result directory or the newest completed study."""
    if override is not None:
        resolved = override.expanduser().resolve()
        if not (resolved / "study_manifest.json").is_file():
            raise FileNotFoundError(f"No study_manifest.json found in {resolved}")
        return resolved

    candidates = sorted(
        (path for path in results_root.glob("*") if (path / "study_manifest.json").is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        available = sorted(path.name for path in results_root.glob("*") if path.is_dir()) if results_root.exists() else []
        raise FileNotFoundError(
            f"No completed parent-selection studies in {results_root}. Available directories: {available or 'none'}"
        )
    return candidates[0].resolve()

RESULT_PATH = resolve_result_directory(RESULT_DIRECTORY, RESULTS_ROOT)
ANALYSIS_PATH = RESULT_PATH / "analysis"
print(f"Analyzing: {RESULT_PATH}")

## Load Benchmark Artifacts

Every available `run_manifest.json` contributes run metadata. Completed trajectories are flattened into a tidy table, preserving the original artifact path for traceability.

In [ ]:
def load_json(path: Path) -> dict:
    """Load a JSON artifact from disk."""
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def repertoire_layout_from_manifest(manifest: dict, manifest_path: Path) -> tuple[tuple[str, ...], tuple[int, ...], int]:
    """Return validated repertoire metadata, including legacy manifest fallbacks."""
    layout = manifest.get("repertoire_layout")
    if isinstance(layout, dict):
        descriptor_names = tuple(str(name) for name in layout.get("descriptor_names", ()))
        n_cells_per_dim = tuple(int(value) for value in layout.get("n_cells_per_dim", ()))
        cell_depth = int(layout.get("cell_depth", 0))
        declared_n_cells = layout.get("n_logical_cells")
    else:
        ga_parameters = manifest.get("parameters", {}).get("ga_config", {})
        descriptor_definitions = ga_parameters.get("me_descriptors", ())
        descriptor_names = tuple(str(definition["metric"]) for definition in descriptor_definitions)
        n_cells_per_dim = tuple(int(definition["num_cells"]) for definition in descriptor_definitions)
        cell_depth = int(ga_parameters.get("cell_depth", 0))
        declared_n_cells = None

    if len(descriptor_names) != len(n_cells_per_dim) or not descriptor_names:
        raise ValueError(f"Invalid repertoire descriptor metadata in {manifest_path}.")
    if len(set(descriptor_names)) != len(descriptor_names) or any(value < 1 for value in n_cells_per_dim):
        raise ValueError(f"Invalid repertoire descriptor layout in {manifest_path}.")
    if cell_depth < 1:
        raise ValueError(f"Invalid repertoire cell depth in {manifest_path}.")

    n_logical_cells = int(np.prod(n_cells_per_dim))
    if declared_n_cells is not None and int(declared_n_cells) != n_logical_cells:
        raise ValueError(f"Repertoire cell count does not match its descriptor layout in {manifest_path}.")
    return descriptor_names, n_cells_per_dim, cell_depth


def parse_repertoire_snapshot(
    record: dict,
    artifact_path: Path,
    n_logical_cells: int,
) -> tuple[int, int, np.ndarray, np.ndarray, np.ndarray]:
    """Validate and normalize one sparse epoch snapshot from a JSONL artifact."""
    try:
        epoch = int(record["epoch"])
        jax_iteration = int(record["jax_iteration"])
        cell_indices = np.asarray(record["cell_indices"], dtype=int).reshape(-1)
        elite_fitnesses = np.asarray(
            [np.nan if value is None else float(value) for value in record["elite_fitnesses"]],
            dtype=float,
        ).reshape(-1)
        selection_counts = np.asarray(record["selection_counts"], dtype=float).reshape(-1)
    except (KeyError, TypeError, ValueError) as error:
        raise ValueError(f"Invalid repertoire snapshot in {artifact_path}.") from error

    if cell_indices.shape != elite_fitnesses.shape or cell_indices.shape != selection_counts.shape:
        raise ValueError(f"Mismatched sparse snapshot arrays in {artifact_path} at epoch {epoch}.")
    if epoch < 0 or jax_iteration < 0:
        raise ValueError(f"Negative epoch metadata in {artifact_path}.")
    if np.any(cell_indices < 0) or np.any(cell_indices >= n_logical_cells):
        raise ValueError(f"Out-of-bounds cell index in {artifact_path} at epoch {epoch}.")
    if np.unique(cell_indices).size != cell_indices.size:
        raise ValueError(f"Duplicate cell index in {artifact_path} at epoch {epoch}.")
    if np.any(~np.isfinite(selection_counts)) or np.any(selection_counts < 0):
        raise ValueError(f"Invalid selection count in {artifact_path} at epoch {epoch}.")
    return epoch, jax_iteration, cell_indices, elite_fitnesses, selection_counts


def load_legacy_archive_snapshot(
    archive_path: Path,
    epoch: int,
    n_logical_cells: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return a final-fitness-only snapshot for a pre-history benchmark artifact."""
    archive_cells = load_json(archive_path)
    if not isinstance(archive_cells, list):
        raise ValueError(f"Expected a list of archive cells in {archive_path}.")

    cell_indices = np.asarray([cell.get("cell_index") for cell in archive_cells], dtype=int)
    elite_fitnesses = np.asarray(
        [np.nan if cell.get("fitness") is None else float(cell["fitness"]) for cell in archive_cells],
        dtype=float,
    )
    if np.any(cell_indices < 0) or np.any(cell_indices >= n_logical_cells):
        raise ValueError(f"Out-of-bounds archive cell index in {archive_path}.")
    if np.unique(cell_indices).size != cell_indices.size:
        raise ValueError(f"Duplicate archive cell index in {archive_path}.")
    order = np.argsort(cell_indices)
    return cell_indices[order], elite_fitnesses[order], np.zeros(cell_indices.size, dtype=float)


def load_artifacts(result_path: Path) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load run metadata, scalar trajectories, and epochwise repertoire snapshots."""
    study = load_json(result_path / "study_manifest.json")
    run_rows: list[dict] = []
    trajectory_rows: list[dict] = []
    repertoire_rows: list[dict] = []
    for manifest_path in sorted((result_path / "runs").glob("**/run_manifest.json")):
        manifest = load_json(manifest_path)
        descriptor_names, n_cells_per_dim, cell_depth = repertoire_layout_from_manifest(manifest, manifest_path)
        grid = manifest.get("grid", {})
        run_row = {
            "run_id": manifest.get("run_id"),
            "grid_id": grid.get("id"),
            "mode": manifest.get("parent_selection_label", manifest.get("parent_selection_mode")),
            "seed": manifest.get("seed"),
            "status": manifest.get("status"),
            "epochs_completed": manifest.get("epochs_completed"),
            "initialization_seconds": manifest.get("phase_seconds", {}).get("initialization"),
            "dc_optimization_seconds": manifest.get("phase_seconds", {}).get("dc_optimization"),
            "manifest_path": str(manifest_path),
            "run_path": str(manifest_path.parent),
            "error": manifest.get("error"),
            "descriptor_names": descriptor_names,
            "n_cells_per_dim": n_cells_per_dim,
            "cell_depth": cell_depth,
            "repertoire_data_source": None,
        }
        trajectory_path = manifest_path.parent / "trajectory.jsonl"
        if manifest.get("status") == "completed" and trajectory_path.is_file():
            with trajectory_path.open(encoding="utf-8") as handle:
                for line in handle:
                    record = json.loads(line)
                    row = pd.json_normalize(record, sep=".").to_dict(orient="records")[0]
                    row.update(
                        {
                            "run_id": run_row["run_id"],
                            "grid_id": run_row["grid_id"],
                            "mode": run_row["mode"],
                            "seed": run_row["seed"],
                            "run_path": run_row["run_path"],
                        }
                    )
                    trajectory_rows.append(row)

        repertoire_path = manifest_path.parent / "repertoire_trajectory.jsonl"
        n_logical_cells = int(np.prod(n_cells_per_dim))
        if manifest.get("status") == "completed" and repertoire_path.is_file():
            with repertoire_path.open(encoding="utf-8") as handle:
                for line in handle:
                    record = json.loads(line)
                    epoch, jax_iteration, cell_indices, elite_fitnesses, selection_counts = parse_repertoire_snapshot(
                        record,
                        repertoire_path,
                        n_logical_cells,
                    )
                    repertoire_rows.append(
                        {
                            "run_id": run_row["run_id"],
                            "grid_id": run_row["grid_id"],
                            "mode": run_row["mode"],
                            "seed": run_row["seed"],
                            "epoch": epoch,
                            "jax_iteration": jax_iteration,
                            "cell_indices": cell_indices,
                            "elite_fitnesses": elite_fitnesses,
                            "selection_counts": selection_counts,
                            "selection_counts_available": True,
                            "snapshot_source": "epoch_trajectory",
                        }
                    )
            run_row["repertoire_data_source"] = "epoch_trajectory"
        else:
            archive_path = manifest_path.parent / "archive_cells.json"
            if manifest.get("status") == "completed" and archive_path.is_file():
                cell_indices, elite_fitnesses, selection_counts = load_legacy_archive_snapshot(
                    archive_path,
                    int(manifest.get("epochs_completed") or 0),
                    n_logical_cells,
                )
                repertoire_rows.append(
                    {
                        "run_id": run_row["run_id"],
                        "grid_id": run_row["grid_id"],
                        "mode": run_row["mode"],
                        "seed": run_row["seed"],
                        "epoch": int(manifest.get("epochs_completed") or 0),
                        "jax_iteration": None,
                        "cell_indices": cell_indices,
                        "elite_fitnesses": elite_fitnesses,
                        "selection_counts": selection_counts,
                        "selection_counts_available": False,
                        "snapshot_source": "archive_cells",
                    }
                )
                run_row["repertoire_data_source"] = "archive_cells"
        run_rows.append(run_row)

    repertoire_snapshots = pd.DataFrame(
        repertoire_rows,
        columns=[
            "run_id",
            "grid_id",
            "mode",
            "seed",
            "epoch",
            "jax_iteration",
            "cell_indices",
            "elite_fitnesses",
            "selection_counts",
            "selection_counts_available",
            "snapshot_source",
        ],
    )
    if not repertoire_snapshots.empty and repertoire_snapshots.duplicated(["run_id", "epoch"]).any():
        raise ValueError("Every run must contain at most one repertoire snapshot per epoch.")
    return study, pd.DataFrame(run_rows), pd.DataFrame(trajectory_rows), repertoire_snapshots


def validate_study_matrix(runs: pd.DataFrame, trajectory: pd.DataFrame) -> tuple[pd.DataFrame, list[int]]:
    """Return planned mode-seed coverage and seeds completed by every mode."""
    if trajectory.empty:
        raise RuntimeError("No completed trajectories were found. Inspect the run-health table for failures.")
    required_columns = [
        FITNESS_COLUMN,
        INITIAL_FITNESS_COLUMN,
        ESSENTIAL_METRICS["overload"],
        ESSENTIAL_METRICS["diversity"],
        ESSENTIAL_METRICS["efficiency"],
        "elapsed_seconds",
        "epoch",
    ]
    missing_columns = [column for column in required_columns if column not in trajectory.columns]
    if missing_columns:
        raise KeyError(f"The selected study is missing required trajectory fields: {missing_columns}")

    modes = [
        mode
        for mode in ["UNIi", "UNIc", "UCBc", "UCBb", "UCBs", "Ec", "Xc", "G"]
        if mode in runs["mode"].dropna().unique()
    ]
    if BASELINE_MODE not in modes:
        raise ValueError(f"Baseline mode {BASELINE_MODE!r} is not present in this study.")
    seeds = sorted(int(seed) for seed in runs["seed"].dropna().unique())
    expected = pd.MultiIndex.from_product([modes, seeds], names=["mode", "seed"]).to_frame(index=False)
    completed = runs.loc[runs["status"].eq("completed"), ["mode", "seed", "run_id"]].drop_duplicates(["mode", "seed"])
    coverage = expected.merge(completed, on=["mode", "seed"], how="left", indicator=True)
    coverage["completed"] = coverage.pop("_merge").eq("both")
    complete_seeds = [
        seed
        for seed, seed_rows in coverage.groupby("seed")
        if bool(seed_rows["completed"].all())
    ]
    if not complete_seeds:
        raise RuntimeError("No seed completed for every parent-selection mode; paired comparisons are unavailable.")
    if len(complete_seeds) < 5:
        warnings.warn(
            f"Only {len(complete_seeds)} fully paired seed(s) are available. Treat uncertainty estimates as exploratory.",
            stacklevel=2,
        )
    return coverage, complete_seeds


study_manifest, runs, trajectory, repertoire_snapshots = load_artifacts(RESULT_PATH)
PAIR_COVERAGE, COMPLETE_SEEDS = validate_study_matrix(runs, trajectory)
MODE_ORDER = [
    mode
    for mode in ["UNIi", "UNIc", "UCBc", "UCBb", "UCBs", "Ec", "Xc", "G"]
    if mode in runs["mode"].dropna().unique()
]
BEST_METRIC_COLUMNS = [column for column in trajectory.columns if column.startswith("quality.best_candidate_metrics.")]
AVAILABLE_BEST_METRICS = sorted(column.removeprefix("quality.best_candidate_metrics.") for column in BEST_METRIC_COLUMNS)
print(
    f"Loaded {len(runs)} runs, {len(trajectory)} scalar epoch records, and {len(repertoire_snapshots)} repertoire snapshots "
    f"across {len(MODE_ORDER)} modes; {len(COMPLETE_SEEDS)} fully paired seed(s): {COMPLETE_SEEDS}."
)

## Study Health and Final Outcomes

In [ ]:
OVERLOAD_COLUMN = metric_column(OVERLOAD_METRIC)
TERMINAL_BRANCH_COLUMN = ESSENTIAL_METRICS["efficiency"]

numeric_columns = [
    "epoch",
    "elapsed_seconds",
    FITNESS_COLUMN,
    INITIAL_FITNESS_COLUMN,
    OVERLOAD_COLUMN,
    ESSENTIAL_METRICS["diversity"],
    TERMINAL_BRANCH_COLUMN,
    metric_column(ESSENTIAL_METRICS["severity"]),
    metric_column(ESSENTIAL_METRICS["criticality"]),
    *(metric_column(metric) for metric in ESSENTIAL_METRICS["operational_cost"]),
]
for column in numeric_columns:
    if column in trajectory.columns:
        trajectory[column] = pd.to_numeric(trajectory[column], errors="coerce")

trajectory = trajectory.sort_values(["run_id", "epoch", "elapsed_seconds"]).reset_index(drop=True)
initial_records = trajectory.groupby("run_id", as_index=False).head(1)[["run_id", FITNESS_COLUMN]].rename(
    columns={FITNESS_COLUMN: INITIAL_FITNESS_COLUMN}
)
final_records = trajectory.groupby("run_id", as_index=False).tail(1)
run_metadata = runs.drop(columns=["mode", "seed", "run_path"], errors="ignore")
finals = (
    final_records
    .merge(initial_records, on="run_id", how="left", suffixes=("", "_recorded"), validate="one_to_one")
    .merge(run_metadata, on="run_id", how="left", validate="one_to_one")
)
finals["fitness_improvement"] = finals[FITNESS_COLUMN] - finals[INITIAL_FITNESS_COLUMN]
finals["terminal_branch_combinations"] = finals[TERMINAL_BRANCH_COLUMN]
finals["terminal_branch_combinations_per_second"] = np.where(
    finals["dc_optimization_seconds"].gt(0),
    finals["terminal_branch_combinations"] / finals["dc_optimization_seconds"],
    np.nan,
)

run_health = runs.merge(
    finals[["run_id", "terminal_branch_combinations_per_second"]].rename(
        columns={"terminal_branch_combinations_per_second": "candidates_per_second"}
    ),
    on="run_id",
    how="left",
    validate="one_to_one",
)

missing_contract_metrics = sorted(
    {
        OVERLOAD_METRIC,
        ESSENTIAL_METRICS["severity"],
        ESSENTIAL_METRICS["criticality"],
        *ESSENTIAL_METRICS["operational_cost"],
    } - set(AVAILABLE_BEST_METRICS)
)
if missing_contract_metrics:
    warnings.warn(
        f"This study lacks configured guardrail metrics: {missing_contract_metrics}. "
        "They will be omitted from the report.",
        stacklevel=2,
    )

run_health_table = (
    run_health.groupby("mode").agg(
        runs=("run_id", "size"),
        completed=("status", lambda values: int(values.eq("completed").sum())),
        epochs_completed_mean=("epochs_completed", "mean"),
        initialization_seconds_mean=("initialization_seconds", "mean"),
        dc_optimization_seconds_mean=("dc_optimization_seconds", "mean"),
        candidates_per_second_mean=("candidates_per_second", "mean"),
        errors=("error", lambda values: int(values.notna().sum())),
    )
    .reindex(MODE_ORDER)
    .reset_index()
)
display(run_health_table)

## Paired Final Fitness Outcome

Every line links one shared seed across modes. Colored markers show the mean final fitness improvement with one sample standard-deviation error bar. The paired table separately reports median fitness differences against `UNIi`. N-1 overload energy and switching distance remain visible as guardrails in the outcome summary.

In [ ]:
def paired_bootstrap_interval(differences: pd.Series) -> tuple[float, float]:
    """Return a deterministic percentile interval for a paired median difference."""
    values = differences.dropna().to_numpy(dtype=float)
    if not len(values):
        return np.nan, np.nan
    if len(values) == 1:
        return float(values[0]), float(values[0])
    random_generator = np.random.default_rng(0)
    samples = random_generator.choice(values, size=(BOOTSTRAP_RESAMPLES, len(values)), replace=True)
    estimates = np.median(samples, axis=1)
    return tuple(float(value) for value in np.quantile(estimates, [0.025, 0.975]))


paired_finals = finals.loc[finals["seed"].isin(COMPLETE_SEEDS)].copy()
baseline_fitness = paired_finals.loc[
    paired_finals["mode"].eq(BASELINE_MODE), ["seed", "fitness_improvement"]
].rename(columns={"fitness_improvement": "baseline_fitness_improvement"})
paired_fitness = paired_finals.merge(baseline_fitness, on="seed", how="inner", validate="many_to_one")
paired_fitness["delta_vs_UNIi"] = (
    paired_fitness["fitness_improvement"] - paired_fitness["baseline_fitness_improvement"]
)
paired_scorecard = (
    paired_fitness.loc[~paired_fitness["mode"].eq(BASELINE_MODE)]
    .groupby("mode", as_index=False)["delta_vs_UNIi"]
    .agg(
        paired_seeds="count",
        median_delta_vs_UNIi="median",
        wins=lambda values: int(values.gt(0).sum()),
        ties=lambda values: int(np.isclose(values, 0.0, rtol=1e-8, atol=1e-10).sum()),
        losses=lambda values: int(values.lt(0).sum()),
    )
)
intervals = paired_scorecard["mode"].map(
    paired_fitness.groupby("mode")["delta_vs_UNIi"].apply(paired_bootstrap_interval)
)
paired_scorecard[["bootstrap_95_low", "bootstrap_95_high"]] = pd.DataFrame(intervals.tolist(), index=paired_scorecard.index)

paired_final_fitness_figure = go.Figure()
for seed, seed_data in paired_fitness.groupby("seed"):
    ordered = seed_data.set_index("mode").reindex(MODE_ORDER).dropna(subset=["fitness_improvement"])
    paired_final_fitness_figure.add_trace(
        go.Scatter(
            x=ordered.index,
            y=ordered["fitness_improvement"],
            mode="lines+markers",
            line={"color": "#8a8a8a", "width": 1},
            marker={"size": 7},
            opacity=0.6,
            showlegend=False,
            hovertemplate=f"Seed {seed}<br>%{{x}}<br>Fitness improvement: %{{y:.4g}}<extra></extra>",
        )
    )
fitness_summary = paired_fitness.groupby("mode")["fitness_improvement"].agg(["mean", "std"]).reindex(MODE_ORDER)
paired_final_fitness_figure.add_trace(
    go.Scatter(
        x=fitness_summary.index,
        y=fitness_summary["mean"],
        error_y={"type": "data", "array": fitness_summary["std"].fillna(0.0), "visible": True},
        mode="markers",
        marker={"color": [PALETTE[mode] for mode in fitness_summary.index], "size": 13, "symbol": "diamond"},
        name="Mean +/- SD",
        hovertemplate="%{x}<br>Mean fitness improvement: %{y:.4g}<br>Sample SD: %{error_y.array:.4g}<extra></extra>",
    )
)
paired_final_fitness_figure.update_layout(
    title="Paired final fitness improvement (mean +/- SD)",
    height=FIGURE_HEIGHT,
    yaxis_title="Fitness improvement from initial state",
    xaxis_title="Parent-selection mode",
)
paired_final_fitness_figure.show()

final_outcome_table = (
    finals.groupby("mode").agg(
        completed_seeds=("run_id", "size"),
        fitness_improvement_mean=("fitness_improvement", "mean"),
        final_best_fitness_mean=(FITNESS_COLUMN, "mean"),
        overload_energy_n_1_mean=(OVERLOAD_COLUMN, "mean"),
        switching_distance_mean=(metric_column("switching_distance"), "mean"),
    )
    .reindex(MODE_ORDER)
    .reset_index()
    .merge(paired_scorecard[["mode", "median_delta_vs_UNIi", "wins", "ties", "losses"]], on="mode", how="left")
)
display(final_outcome_table)

## Fitness Improvement Trajectories


In [ ]:
progress = trajectory.loc[trajectory["seed"].isin(COMPLETE_SEEDS)].copy()
progress[INITIAL_FITNESS_COLUMN] = progress["run_id"].map(initial_records.set_index("run_id")[INITIAL_FITNESS_COLUMN])
progress["fitness_improvement"] = progress[FITNESS_COLUMN] - progress[INITIAL_FITNESS_COLUMN]
progress["epoch"] = pd.to_numeric(progress["epoch"], errors="coerce")
progress["elapsed_seconds"] = pd.to_numeric(progress["elapsed_seconds"], errors="coerce")


def add_fitness_trajectory_traces(
    figure: go.Figure,
    mode: str,
    mode_data: pd.DataFrame,
    axis: str,
    axis_label: str,
) -> None:
    """Add observed per-seed and median fitness-improvement trajectories for one mode."""
    if SHOW_SEED_TRACES:
        for seed, seed_data in mode_data.groupby("seed"):
            figure.add_trace(
                go.Scatter(
                    x=seed_data[axis],
                    y=seed_data["fitness_improvement"],
                    mode="lines+markers",
                    line={"color": PALETTE[mode], "width": 1},
                    marker={"size": 5},
                    opacity=0.18,
                    showlegend=False,
                    hovertemplate=(
                        f"{mode}<br>Seed: {seed}<br>{axis_label}: %{{x:.4g}}"
                        + "<br>Fitness improvement: %{y:.4g}<extra></extra>"
                    ),
                )
            )
    summary = mode_data.groupby("epoch", as_index=False).agg(
        elapsed_seconds=("elapsed_seconds", "median"),
        fitness_improvement=("fitness_improvement", "median"),
        q25=("fitness_improvement", lambda values: values.quantile(0.25)),
        q75=("fitness_improvement", lambda values: values.quantile(0.75)),
    )
    figure.add_trace(
        go.Scatter(
            x=pd.concat([summary[axis], summary[axis].iloc[::-1]]),
            y=pd.concat([summary["q75"], summary["q25"].iloc[::-1]]),
            fill="toself",
            fillcolor=transparent_color(PALETTE[mode]),
            line={"width": 0},
            hoverinfo="skip",
            showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=summary[axis],
            y=summary["fitness_improvement"],
            mode="lines+markers",
            line={"color": PALETTE[mode], "width": 3},
            marker={"size":7},
            name=mode,
            hovertemplate=f"{mode}<br>{axis_label}: %{{x:.4g}}<br>Median fitness improvement: %{{y:.4g}}<extra></extra>",
        )
    )


fitness_by_epoch_figure = go.Figure()
fitness_by_time_figure = go.Figure()
for mode in MODE_ORDER:
    mode_data = progress.loc[progress["mode"].eq(mode)].sort_values(["seed", "epoch"])
    add_fitness_trajectory_traces(
        fitness_by_epoch_figure,
        mode,
        mode_data,
        axis="epoch",
        axis_label="Completed epoch",
    )
    add_fitness_trajectory_traces(
        fitness_by_time_figure,
        mode,
        mode_data,
        axis="elapsed_seconds",
        axis_label="Elapsed DC optimization time (s)",
    )

fitness_by_epoch_figure.update_layout(
    title="Observed fitness improvement by completed epoch",
    height=FIGURE_HEIGHT,
    xaxis_title="Completed optimization epoch",
    yaxis_title="Fitness improvement from initial state",
    legend_title_text="Mode",
)
fitness_by_epoch_figure.show()

fitness_by_time_figure.update_layout(
    title="Observed fitness improvement by elapsed optimization time",
    height=FIGURE_HEIGHT,
    xaxis_title="DC optimization wall-clock time after initialization (s)",
    yaxis_title="Fitness improvement from initial state",
    legend_title_text="Mode",
)
fitness_by_time_figure.show()

## Final Quality-Cost Trade-Off

The paired panels show final N-1 and N-0 overload against switching distance. Hover details retain the number of split substations and configured fitness, so both contingency and base-case outcomes remain visible with operational cost.

In [ ]:
n_0_overload_column = metric_column("overload_energy_n_0")
tradeoff_columns = [OVERLOAD_COLUMN, n_0_overload_column, metric_column("switching_distance"), metric_column("split_subs")]
tradeoff_data = paired_finals.dropna(subset=tradeoff_columns).copy()
tradeoff_figure: go.Figure | None = None
if tradeoff_data.empty:
    print("Final N-1/N-0 quality-cost trade-off metrics are not available for this study.")
else:
    tradeoff_data[tradeoff_columns] = tradeoff_data[tradeoff_columns].apply(pd.to_numeric, errors="coerce")
    tradeoff_data = tradeoff_data.dropna(subset=tradeoff_columns)

    def padded_axis_limits(values: pd.Series) -> tuple[float, float]:
        """Return readable limits when a metric has little or no spread."""
        minimum, maximum = float(values.min()), float(values.max())
        padding = max((maximum - minimum) * 0.1, abs(values.mean()) * 0.005, 1.0)
        return minimum - padding, maximum + padding

    overload_panels = [
        (OVERLOAD_COLUMN, "Final N-1 overload energy (MW)", "N-1 overload"),
        (n_0_overload_column, "Final N-0 overload energy (MW)", "N-0 overload"),
    ]
    tradeoff_figure = make_subplots(
        rows=1,
        cols=2,
        shared_xaxes=True,
        subplot_titles=[panel[2] for panel in overload_panels],
        horizontal_spacing=0.1,
    )
    for panel_index, (overload_column, overload_label, _) in enumerate(overload_panels, start=1):
        for mode_index, mode in enumerate(MODE_ORDER):
            mode_data = tradeoff_data.loc[tradeoff_data["mode"].eq(mode)]
            if mode_data.empty:
                continue
            tradeoff_figure.add_trace(
                go.Scatter(
                    x=mode_data[tradeoff_columns[2]],
                    y=mode_data[overload_column],
                    customdata=mode_data[["seed", tradeoff_columns[3], "fitness_improvement", FITNESS_COLUMN, "run_path"]],
                    mode="markers",
                    marker={
                        "color": PALETTE[mode],
                        "size": 14,
                        "symbol": ["circle", "diamond", "square", "x", "triangle-up", "cross", "star", "hexagon"][mode_index],
                    },
                    name=mode,
                    legendgroup=mode,
                    showlegend=panel_index == 1,
                    hovertemplate=(
                        f"{mode}<br>Switching distance: %{{x}}<br>{overload_label}: %{{y:.4g}}"
                        + "<br>Seed: %{customdata[0]}<br>Split substations: %{customdata[1]}"
                        + "<br>Fitness improvement: %{customdata[2]:.4g}<br>Final fitness: %{customdata[3]:.4g}"
                        + "<br>Run path: %{customdata[4]}<extra></extra>"
                    ),
                ),
                row=1,
                col=panel_index,
            )
        axis_minimum, axis_maximum = padded_axis_limits(tradeoff_data[overload_column])
        tradeoff_figure.update_xaxes(title_text="Switching distance (reassignment cost)", row=1, col=panel_index)
        tradeoff_figure.update_yaxes(
            title_text=overload_label,
            range=[axis_minimum, axis_maximum],
            nticks=6,
            tickformat=".2f",
            row=1,
            col=panel_index,
        )
        if np.isclose(axis_minimum, axis_maximum, rtol=1e-9, atol=1e-9):
            tradeoff_figure.add_annotation(
                text="All modes overlap at displayed precision.",
                xref=f"x{panel_index} domain" if panel_index > 1 else "x domain",
                yref=f"y{panel_index} domain" if panel_index > 1 else "y domain",
                x=0.5,
                y=1.08,
                showarrow=False,
            )
    tradeoff_figure.update_layout(
        title="Final contingency and base-case overload versus operational cost",
        height=max(FIGURE_HEIGHT, 620),
        legend_title_text="Mode",
    )
    tradeoff_figure.show()

## Repertoire Analyzer

Choose a grid, parent-selection mode, seed aggregation, metric, epoch, and two to four repertoire descriptors. The analyzer renders every selected descriptor pair with shared color scaling.

In [ ]:
ALL_SEEDS = "__all_seeds__"
REPERTOIRE_METRICS = {
    "fitness": {
        "label": "Fitness",
        "colorbar_label": "Fitness (signed log scale)",
        "colorscale": [[0.0, "#d95d73"], [0.45, "#f2b880"], [0.7, "#78bfa8"], [1.0, "#0f766e"]],
        "color_transform": "signed_log",
    },
    "selection_count": {
        "label": "Cumulative selection count",
        "colorbar_label": "Cumulative selection count",
        "colorscale": [[0.0, "#e0f2fe"], [0.35, "#67e8f9"], [0.7, "#0891b2"], [1.0, "#164e63"]],
        "color_transform": "linear",
    },
}
REPERTOIRE_EMPTY_CELL_COLOR = "#e2e8f0"
repertoire_analyzer_figure: go.Figure | None = None
repertoire_color_range_cache: dict[tuple[tuple[str, ...], tuple[tuple[str, str], ...], str], tuple[float, float]] = {}


def selected_repertoire_runs(grid_id: str, mode: str, seed_value: str | int) -> tuple[pd.DataFrame, tuple[str, ...], tuple[int, ...]]:
    """Return same-layout completed runs selected for one repertoire view."""
    matching_runs = runs.loc[
        runs["status"].eq("completed")
        & runs["grid_id"].eq(grid_id)
        & runs["mode"].eq(mode)
        & runs["repertoire_data_source"].notna()
    ].copy()
    if seed_value != ALL_SEEDS:
        matching_runs = matching_runs.loc[matching_runs["seed"].eq(seed_value)]
    if matching_runs.empty:
        raise ValueError("No repertoire artifacts are available for the current grid, mode, and seed selection.")
    layouts = matching_runs[["descriptor_names", "n_cells_per_dim"]].drop_duplicates()
    if len(layouts) != 1:
        raise ValueError("Selected runs do not share one descriptor layout and cannot be aggregated.")
    layout = layouts.iloc[0]
    return matching_runs, tuple(layout["descriptor_names"]), tuple(layout["n_cells_per_dim"])


def available_repertoire_epochs(selected_runs: pd.DataFrame) -> tuple[int, ...]:
    """Return all recorded epochs for selected runs."""
    epochs = repertoire_snapshots.loc[repertoire_snapshots["run_id"].isin(selected_runs["run_id"]), "epoch"]
    return tuple(sorted(int(epoch) for epoch in epochs.unique()))


def default_repertoire_epoch(selected_runs: pd.DataFrame, seed_value: str | int) -> int:
    """Return the final epoch of one seed or the last epoch shared by all seeds."""
    epochs = available_repertoire_epochs(selected_runs)
    if not epochs:
        raise ValueError("No repertoire epochs are available for the current selection.")
    if seed_value != ALL_SEEDS:
        return epochs[-1]
    run_epochs = [
        set(repertoire_snapshots.loc[repertoire_snapshots["run_id"].eq(run_id), "epoch"])
        for run_id in selected_runs["run_id"]
    ]
    common_epochs = set.intersection(*run_epochs)
    return max(common_epochs) if common_epochs else epochs[-1]


def selected_epoch_projections(
    selected_runs: pd.DataFrame,
    descriptor_names: tuple[str, ...],
    n_cells_per_dim: tuple[int, ...],
    selected_pairs: tuple[tuple[str, str], ...],
    metric: str,
    epoch: int,
) -> tuple[dict[tuple[str, str], np.ndarray], pd.DataFrame]:
    """Project all available selected runs at one epoch onto each visible descriptor pair."""
    snapshots = repertoire_snapshots.loc[
        repertoire_snapshots["run_id"].isin(selected_runs["run_id"])
        & repertoire_snapshots["epoch"].eq(epoch)
    ].copy()
    if snapshots.empty:
        raise ValueError("No selected run reached the chosen epoch.")
    projections: dict[tuple[str, str], np.ndarray] = {}
    for descriptor_pair in selected_pairs:
        seed_projections = [
            project_repertoire_snapshot(
                cell_indices=snapshot.cell_indices,
                metric_values=snapshot.elite_fitnesses if metric == "fitness" else snapshot.selection_counts,
                n_cells_per_dim=n_cells_per_dim,
                descriptor_names=descriptor_names,
                descriptor_pair=descriptor_pair,
                metric=metric,
            )
            for snapshot in snapshots.itertuples(index=False)
        ]
        projections[descriptor_pair] = aggregate_seed_projections(seed_projections, metric=metric)
    return projections, snapshots


def transform_repertoire_color_values(values: np.ndarray, metric: str) -> np.ndarray:
    """Return values transformed only for perceptually useful color interpolation."""
    values = np.asarray(values, dtype=float)
    if REPERTOIRE_METRICS[metric]["color_transform"] == "signed_log":
        return np.sign(values) * np.log1p(np.abs(values))
    return values


def inverse_repertoire_color_values(values: np.ndarray, metric: str) -> np.ndarray:
    """Return original metric values from transformed color-axis values."""
    values = np.asarray(values, dtype=float)
    if REPERTOIRE_METRICS[metric]["color_transform"] == "signed_log":
        return np.sign(values) * np.expm1(np.abs(values))
    return values


def color_range_from_projections(
    projections: dict[tuple[str, str], np.ndarray],
    metric: str,
) -> tuple[float, float]:
    """Return stable transformed Plotly color bounds for finite projection values."""
    finite_values = [matrix[np.isfinite(matrix)] for matrix in projections.values() if np.isfinite(matrix).any()]
    if not finite_values:
        return 0.0, 1.0
    color_values = transform_repertoire_color_values(np.concatenate(finite_values), metric)
    color_min, color_max = float(color_values.min()), float(color_values.max())
    if np.isclose(color_min, color_max):
        return color_min - 0.5, color_max + 0.5
    return color_min, color_max


def repertoire_color_range(
    selected_runs: pd.DataFrame,
    descriptor_names: tuple[str, ...],
    n_cells_per_dim: tuple[int, ...],
    selected_pairs: tuple[tuple[str, str], ...],
    metric: str,
) -> tuple[float, float]:
    """Return cached color bounds across every available epoch of the current view."""
    cache_key = (tuple(sorted(selected_runs["run_id"])), selected_pairs, metric)
    if cache_key not in repertoire_color_range_cache:
        all_epoch_projections: dict[tuple[str, str], np.ndarray] = {}
        for epoch in available_repertoire_epochs(selected_runs):
            epoch_projections, _snapshots = selected_epoch_projections(
                selected_runs=selected_runs,
                descriptor_names=descriptor_names,
                n_cells_per_dim=n_cells_per_dim,
                selected_pairs=selected_pairs,
                metric=metric,
                epoch=epoch,
            )
            all_epoch_projections.update({(pair[0], f"{pair[1]}@{epoch}"): matrix for pair, matrix in epoch_projections.items()})
        repertoire_color_range_cache[cache_key] = color_range_from_projections(all_epoch_projections, metric)
    return repertoire_color_range_cache[cache_key]


def colorbar_ticks(color_range: tuple[float, float], metric: str) -> tuple[list[float], list[str]]:
    """Return transformed tick positions labelled with original metric values."""
    tick_values = np.linspace(*color_range, num=5)
    original_values = inverse_repertoire_color_values(tick_values, metric)
    return tick_values.tolist(), [f"{value:.4g}" for value in original_values]


def plot_repertoire_projections(
    projections: dict[tuple[str, str], np.ndarray],
    metric: str,
    grid_id: str,
    mode: str,
    seed_label: str,
    epoch: int,
    n_seeds: int,
    color_range: tuple[float, float],
) -> go.Figure:
    """Render descriptor-pair heatmaps with square cells and epoch-stable color scaling."""
    metric_settings = REPERTOIRE_METRICS[metric]
    pairs = tuple(projections)
    n_columns = 1 if len(pairs) == 1 else 3
    n_rows = int(np.ceil(len(pairs) / n_columns))
    figure = make_subplots(
        rows=n_rows,
        cols=n_columns,
        subplot_titles=[f"{vertical} x {horizontal}" for vertical, horizontal in pairs],
        horizontal_spacing=0.08,
        vertical_spacing=0.16,
    )
    color_min, color_max = color_range
    colorbar_tick_values, colorbar_tick_labels = colorbar_ticks(color_range, metric)
    has_finite_values = any(np.isfinite(matrix).any() for matrix in projections.values())

    for plot_index, ((vertical, horizontal), matrix) in enumerate(projections.items()):
        row, column = divmod(plot_index, n_columns)
        x_coordinates = np.arange(matrix.shape[1])
        y_coordinates = np.arange(matrix.shape[0])
        color_matrix = transform_repertoire_color_values(matrix, metric)
        figure.add_trace(
            go.Heatmap(
                x=x_coordinates,
                y=y_coordinates,
                z=np.ones(matrix.shape, dtype=float),
                colorscale=[[0.0, REPERTOIRE_EMPTY_CELL_COLOR], [1.0, REPERTOIRE_EMPTY_CELL_COLOR]],
                showscale=False,
                hoverinfo="skip",
                xgap=1,
                ygap=1,
            ),
            row=row + 1,
            col=column + 1,
        )
        figure.add_trace(
            go.Heatmap(
                x=x_coordinates,
                y=y_coordinates,
                z=color_matrix,
                customdata=matrix,
                coloraxis="coloraxis",
                hovertemplate=(
                    f"{vertical}: %{{y}}<br>{horizontal}: %{{x}}<br>"
                    f"{metric_settings['label']}: %{{customdata:.4g}}<extra></extra>"
                ),
                hoverongaps=False,
                xgap=1,
                ygap=1,
            ),
            row=row + 1,
            col=column + 1,
        )
        xaxis_reference = "x" if plot_index == 0 else f"x{plot_index + 1}"
        figure.update_xaxes(title_text=horizontal, constrain="domain", row=row + 1, col=column + 1)
        figure.update_yaxes(
            title_text=vertical,
            scaleanchor=xaxis_reference,
            scaleratio=1,
            constrain="domain",
            row=row + 1,
            col=column + 1,
        )

    figure.update_layout(
        title=(
            f"Repertoire analyzer: {metric_settings['label']}<br>"
            f"<sup>Grid {grid_id} | Mode {mode} | {seed_label} | Epoch {epoch} | n={n_seeds}</sup>"
        ),
        width=max(FIGURE_WIDTH, 400 * n_columns),
        height=max(FIGURE_HEIGHT, 360 * n_rows),
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        coloraxis={
            "colorscale": metric_settings["colorscale"],
            "cmin": color_min,
            "cmax": color_max,
            "colorbar": {
                "title": metric_settings["colorbar_label"],
                "tickmode": "array",
                "tickvals": colorbar_tick_values,
                "ticktext": colorbar_tick_labels,
            },
        },
    )
    if not has_finite_values:
        figure.add_annotation(
            text="No occupied repertoire cells at this epoch.",
            x=0.5,
            y=0.5,
            xref="paper",
            yref="paper",
            showarrow=False,
        )
    return figure


def build_repertoire_analyzer() -> widgets.Widget:
    """Build controls and output for epochwise repertoire analysis."""
    available_grids = sorted(runs.loc[runs["repertoire_data_source"].notna(), "grid_id"].dropna().unique())
    if not available_grids:
        return widgets.HTML("No repertoire artifacts are available in this study.")

    grid_dropdown = widgets.Dropdown(description="Grid", options=available_grids, value=available_grids[0])
    mode_dropdown = widgets.Dropdown(description="Mode")
    seed_dropdown = widgets.Dropdown(description="Seed")
    metric_dropdown = widgets.Dropdown(
        description="Metric",
        options=[(settings["label"], metric) for metric, settings in REPERTOIRE_METRICS.items()],
        value="fitness",
    )
    descriptor_dropdown = widgets.Accordion(children=(widgets.VBox(),), layout=widgets.Layout(width="320px"))
    descriptor_checkboxes: dict[str, widgets.Checkbox] = {}
    epoch_slider = widgets.SelectionSlider(description="Epoch", options=(0,), value=0, continuous_update=False)
    output = widgets.Output()
    updating = False

    def show_message(message: str) -> None:
        with output:
            output.clear_output(wait=True)
            print(message)

    def selected_descriptor_names(descriptor_names: tuple[str, ...]) -> tuple[str, ...]:
        """Return checked descriptors in configured repertoire order."""
        return tuple(name for name in descriptor_names if descriptor_checkboxes.get(name) is not None and descriptor_checkboxes[name].value)

    def update_descriptor_checkboxes(descriptor_names: tuple[str, ...]) -> None:
        """Rebuild the descriptor dropdown while preserving valid checked names."""
        selected_descriptors = selected_descriptor_names(descriptor_names)
        if not 2 <= len(selected_descriptors) <= 4:
            selected_descriptors = tuple(descriptor_names[: min(4, len(descriptor_names))])
        selected_names = set(selected_descriptors)
        descriptor_checkboxes.clear()
        for name in descriptor_names:
            checkbox = widgets.Checkbox(value=name in selected_names, description=name, indent=False)
            checkbox.observe(render, names="value")
            descriptor_checkboxes[name] = checkbox
        descriptor_dropdown.children = (widgets.VBox(tuple(descriptor_checkboxes.values())),)
        descriptor_dropdown.set_title(0, f"Descriptors ({len(selected_descriptors)} selected)")

    def refresh_controls(_change: dict | None = None) -> None:
        nonlocal updating
        if updating:
            return
        updating = True
        try:
            available_modes = [
                mode
                for mode in MODE_ORDER
                if (
                    runs["grid_id"].eq(grid_dropdown.value)
                    & runs["mode"].eq(mode)
                    & runs["repertoire_data_source"].notna()
                ).any()
            ]
            if not available_modes:
                show_message("No repertoire artifacts are available for the selected grid.")
                return
            mode_dropdown.options = available_modes
            if mode_dropdown.value not in available_modes:
                mode_dropdown.value = BASELINE_MODE if BASELINE_MODE in available_modes else available_modes[0]
            mode_runs, descriptor_names, _n_cells_per_dim = selected_repertoire_runs(
                grid_dropdown.value,
                mode_dropdown.value,
                ALL_SEEDS,
            )
            seed_options: list[tuple[str, str | int]] = [("Mean over all seeds", ALL_SEEDS)]
            seed_options.extend((f"Seed {int(seed)}", int(seed)) for seed in sorted(mode_runs["seed"].unique()))
            seed_dropdown.options = seed_options
            if seed_dropdown.value not in [value for _label, value in seed_options]:
                seed_dropdown.value = ALL_SEEDS
            selected_runs, descriptor_names, _n_cells_per_dim = selected_repertoire_runs(
                grid_dropdown.value,
                mode_dropdown.value,
                seed_dropdown.value,
            )
            update_descriptor_checkboxes(descriptor_names)
            available_epochs = available_repertoire_epochs(selected_runs)
            if not available_epochs:
                show_message("No repertoire epochs are available for the selected runs.")
                return
            epoch_slider.options = available_epochs
            if epoch_slider.value not in available_epochs:
                epoch_slider.value = default_repertoire_epoch(selected_runs, seed_dropdown.value)
        except ValueError as error:
            show_message(str(error))
        finally:
            updating = False
        render()

    def render(_change: dict | None = None) -> None:
        global repertoire_analyzer_figure
        if updating:
            return
        try:
            selected_runs, descriptor_names, n_cells_per_dim = selected_repertoire_runs(
                grid_dropdown.value,
                mode_dropdown.value,
                seed_dropdown.value,
            )
            selected_descriptors = selected_descriptor_names(descriptor_names)
            descriptor_dropdown.set_title(0, f"Descriptors ({len(selected_descriptors)} selected)")
            selected_pairs = descriptor_pairs(selected_descriptors)
            metric = metric_dropdown.value
            selected_epoch = int(epoch_slider.value)
            projections, snapshots = selected_epoch_projections(
                selected_runs=selected_runs,
                descriptor_names=descriptor_names,
                n_cells_per_dim=n_cells_per_dim,
                selected_pairs=selected_pairs,
                metric=metric,
                epoch=selected_epoch,
            )
            if metric == "selection_count" and not bool(snapshots["selection_counts_available"].all()):
                raise ValueError("Cumulative selection counts are unavailable for legacy archive-only runs. Rerun the benchmark.")
            color_range = repertoire_color_range(
                selected_runs=selected_runs,
                descriptor_names=descriptor_names,
                n_cells_per_dim=n_cells_per_dim,
                selected_pairs=selected_pairs,
                metric=metric,
            )
            seed_label = (
                f"Mean over {len(snapshots)} available seed(s)"
                if seed_dropdown.value == ALL_SEEDS
                else f"Seed {int(seed_dropdown.value)}"
            )
            repertoire_analyzer_figure = plot_repertoire_projections(
                projections=projections,
                metric=metric,
                grid_id=grid_dropdown.value,
                mode=mode_dropdown.value,
                seed_label=seed_label,
                epoch=selected_epoch,
                n_seeds=len(snapshots),
                color_range=color_range,
            )
        except ValueError as error:
            repertoire_analyzer_figure = None
            show_message(str(error))
            return
        with output:
            output.clear_output(wait=True)
            display(repertoire_analyzer_figure)

    grid_dropdown.observe(refresh_controls, names="value")
    mode_dropdown.observe(refresh_controls, names="value")
    seed_dropdown.observe(refresh_controls, names="value")
    metric_dropdown.observe(render, names="value")
    epoch_slider.observe(render, names="value")
    refresh_controls()
    top_controls = [mode_dropdown, seed_dropdown, metric_dropdown]
    if len(available_grids) > 1:
        top_controls.insert(0, grid_dropdown)
    return widgets.VBox(
        [
            widgets.HBox(top_controls),
            widgets.HBox([descriptor_dropdown, epoch_slider]),
            output,
        ]
    )


repertoire_analyzer = build_repertoire_analyzer()
display(repertoire_analyzer)

## Export Analysis Artifacts

Set `EXPORT_ARTIFACTS = True` to write normalized tables under the selected study's `analysis/` directory. Set `EXPORT_MARKDOWN_REPORT = True` to create `benchmark_report.md`, a self-contained report with embedded plot images and key result tables.

In [ ]:
import base64
import importlib


def format_markdown_value(value: object) -> str:
    """Format a scalar value for a compact Markdown table."""
    if pd.isna(value):
        return ""
    if isinstance(value, (float, np.floating)):
        return f"{value:.4g}"
    return str(value).replace("|", "\\|").replace("\n", "<br>")


def dataframe_to_markdown(data: pd.DataFrame) -> str:
    """Serialize a DataFrame as Markdown without optional dependencies."""
    table = data.copy()
    if not isinstance(table.index, pd.RangeIndex) or table.index.name is not None:
        table = table.reset_index()
    table.columns = [" ".join(str(part) for part in column if str(part)) if isinstance(column, tuple) else str(column) for column in table.columns]
    rows = [[format_markdown_value(value) for value in row] for row in table.itertuples(index=False, name=None)]
    return "\n".join(
        [
            "| " + " | ".join(table.columns) + " |",
            "| " + " | ".join("---" for _ in table.columns) + " |",
            *("| " + " | ".join(row) + " |" for row in rows),
        ]
    )


def mean_and_std(values: pd.Series) -> str:
    """Format numeric values as mean plus or minus sample standard deviation."""
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return ""
    return f"{values.mean():.4g} +/- {values.std(ddof=1) if len(values) > 1 else 0.0:.3g}"


def figure_to_markdown(heading: str, description: str, figure: go.Figure) -> str:
    """Embed a Plotly figure as a PNG data URI in a Markdown report."""
    import plotly.io._kaleido as plotly_kaleido

    if plotly_kaleido.scope is None:
        importlib.reload(plotly_kaleido)
    try:
        image = figure.to_image(format="png", width=1600, scale=1)
    except ValueError as error:
        raise RuntimeError("Static report export requires Kaleido. Run `uv sync --group dev` and rerun this cell.") from error
    data_uri = base64.b64encode(image).decode("ascii")
    return f"### {heading}\n\n{description}\n\n<img alt=\"{heading}\" src=\"data:image/png;base64,{data_uri}\" />"


def repertoire_snapshot_cell_table(snapshots: pd.DataFrame) -> pd.DataFrame:
    """Expand sparse epoch snapshots into one portable row per logical cell."""
    rows: list[dict[str, object]] = []
    for snapshot in snapshots.itertuples(index=False):
        cell_indices = np.asarray(snapshot.cell_indices, dtype=int).reshape(-1)
        elite_fitnesses = np.asarray(snapshot.elite_fitnesses, dtype=float).reshape(-1)
        selection_counts = np.asarray(snapshot.selection_counts, dtype=float).reshape(-1)
        if cell_indices.shape != elite_fitnesses.shape or cell_indices.shape != selection_counts.shape:
            raise ValueError(f"Mismatched sparse snapshot arrays for run {snapshot.run_id} at epoch {snapshot.epoch}.")
        rows.extend(
            {
                "run_id": snapshot.run_id,
                "grid_id": snapshot.grid_id,
                "mode": snapshot.mode,
                "seed": snapshot.seed,
                "epoch": snapshot.epoch,
                "jax_iteration": snapshot.jax_iteration,
                "cell_index": int(cell_index),
                "elite_fitness": float(elite_fitness),
                "cumulative_selection_count": float(selection_count),
                "selection_counts_available": snapshot.selection_counts_available,
                "snapshot_source": snapshot.snapshot_source,
            }
            for cell_index, elite_fitness, selection_count in zip(cell_indices, elite_fitnesses, selection_counts, strict=True)
        )
    return pd.DataFrame(
        rows,
        columns=[
            "run_id",
            "grid_id",
            "mode",
            "seed",
            "epoch",
            "jax_iteration",
            "cell_index",
            "elite_fitness",
            "cumulative_selection_count",
            "selection_counts_available",
            "snapshot_source",
        ],
    )


def report_configuration() -> pd.DataFrame:
    """Return shared study settings from one completed run manifest."""
    manifest_path = Path(runs.loc[runs["status"].eq("completed"), "manifest_path"].iloc[0])
    manifest = load_json(manifest_path)
    ga, solver, grid = manifest["parameters"]["ga_config"], manifest["parameters"]["loadflow_solver_config"], manifest["grid"]
    targets = ", ".join(f"{metric}: {weight:g}" for metric, weight in ga["target_metrics"])
    descriptors = ", ".join(f"{item['metric']} ({item['num_cells']} cells)" for item in ga["me_descriptors"])
    return pd.DataFrame(
        [
            ("Study status", study_manifest.get("status", "")),
            ("Grid / backend", f"{grid['id']} / {grid['grid_type']}"),
            ("Modes", ", ".join(MODE_ORDER)),
            ("Seeds", ", ".join(map(str, study_manifest.get("seeds", [])))),
            ("Planned / completed runs", f"{study_manifest.get('n_planned_runs', '')} / {len(study_manifest.get('completed_runs', []))}"),
            ("Fully paired seeds", ", ".join(map(str, COMPLETE_SEEDS))),
            ("GA runtime per seed (s)", ga["runtime_seconds"]),
            ("Iterations per epoch", ga["iterations_per_epoch"]),
            ("Target metrics (weight)", targets),
            ("Repertoire descriptors", descriptors),
            ("Cell depth", ga["cell_depth"]),
            ("Loadflow batch size", solver["batch_size"]),
            ("Maximum splits / disconnections", f"{solver['max_num_splits']} / {solver['max_num_disconnections']}"),
        ],
        columns=["setting", "value"],
    )


def summarize_run_health() -> pd.DataFrame:
    """Return one seed-aggregated health row per parent-selection mode."""
    rows = []
    for mode in MODE_ORDER:
        mode_runs = run_health.loc[run_health["mode"].eq(mode)]
        rows.append(
            {
                "mode": mode,
                "runs": len(mode_runs),
                "completed": int(mode_runs["status"].eq("completed").sum()),
                "epochs completed mean +/- std": mean_and_std(mode_runs["epochs_completed"]),
                "initialization seconds mean +/- std": mean_and_std(mode_runs["initialization_seconds"]),
                "DC optimization seconds mean +/- std": mean_and_std(mode_runs["dc_optimization_seconds"]),
                "candidates / second mean +/- std": mean_and_std(mode_runs["candidates_per_second"]),
                "errors": int(mode_runs["error"].notna().sum()) or "none",
            }
        )
    return pd.DataFrame(rows)


def summarize_final_fitness() -> pd.DataFrame:
    """Return final best-fitness improvement summaries by mode across seeds."""
    rows = []
    for mode in MODE_ORDER:
        mode_finals = finals.loc[finals["mode"].eq(mode)]
        paired_delta = paired_fitness.loc[paired_fitness["mode"].eq(mode), "delta_vs_UNIi"]
        rows.append(
            {
                "mode": mode,
                "completed seeds": len(mode_finals),
                "fitness improvement mean +/- std": mean_and_std(mode_finals["fitness_improvement"]),
                "final best fitness mean +/- std": mean_and_std(mode_finals[FITNESS_COLUMN]),
                "paired delta vs UNIi mean +/- std": "-" if mode == BASELINE_MODE else mean_and_std(paired_delta),
            }
        )
    return pd.DataFrame(rows)


def write_markdown_report(report_path: Path) -> None:
    """Write a self-contained benchmark report with configuration, summaries, and figures."""
    figures = [
        ("Paired final fitness improvement", "Each line connects one shared seed across modes; diamonds mark the median final improvement. Higher values are better.", paired_final_fitness_figure),
        ("Observed fitness improvement by epoch", "Archive-best fitness is monotone because archive entries are replaced only by strictly better candidates. Lines show the median at each completed epoch; shading spans the interquartile range across paired seeds.", fitness_by_epoch_figure),
        ("Observed fitness improvement by elapsed time", "The same direct observations are shown against DC optimization wall-clock time. Median points use the median elapsed time and fitness improvement at each completed epoch; no time-grid interpolation is applied.", fitness_by_time_figure),
        ("Final quality-cost trade-off", "Each point is a completed seed. Lower N-0 and N-1 overload energy is better, while switching distance shows the operational cost of the topology change.", tradeoff_figure),
        ("Repertoire analyzer", "The displayed descriptor-pair projections use the current grid, mode, seed aggregation, metric, and epoch selection.", repertoire_analyzer_figure),
    ]
    sections = [
        "# Parent-Selection Benchmark Report", "", f"Study: `{RESULT_PATH}`", "", "## Run Configuration", "", dataframe_to_markdown(report_configuration()), "",
        "## Run Health by Mode", "", "Values are aggregated across seeds and reported as mean +/- sample standard deviation.", "", dataframe_to_markdown(summarize_run_health()), "",
        "## Final Best Fitness Improvement", "", "Values are aggregated across completed seeds. The paired delta uses only seeds completed by every mode.", "", dataframe_to_markdown(summarize_final_fitness()), "", "## Figures",
    ]
    for heading, description, figure in figures:
        if figure is not None:
            sections.extend(["", figure_to_markdown(heading, description, figure)])
    report_path.write_text("\n".join(sections) + "\n", encoding="utf-8")


if EXPORT_ARTIFACTS:
    ANALYSIS_PATH.mkdir(exist_ok=True)
    finals.to_csv(ANALYSIS_PATH / "final_run_metrics.csv", index=False)
    trajectory.to_csv(ANALYSIS_PATH / "trajectory_metrics.csv", index=False)
    paired_scorecard.to_csv(ANALYSIS_PATH / "paired_fitness_scorecard.csv", index=False)
    progress.to_csv(ANALYSIS_PATH / "epoch_trajectories.csv", index=False)
    if not repertoire_snapshots.empty:
        repertoire_snapshot_cell_table(repertoire_snapshots).to_csv(
            ANALYSIS_PATH / "repertoire_snapshot_cells.csv",
            index=False,
        )
    (ANALYSIS_PATH / "analysis_metadata.json").write_text(
        json.dumps(
            {
                "result_directory": str(RESULT_PATH),
                "baseline_mode": BASELINE_MODE,
                "fitness_column": FITNESS_COLUMN,
                "overload_guardrail_metric": OVERLOAD_METRIC,
                "complete_seeds": COMPLETE_SEEDS,
                "show_seed_traces": SHOW_SEED_TRACES,
                "modes": MODE_ORDER,
                "repertoire_snapshots_available": not repertoire_snapshots.empty,
                "selection_count_history_available": bool(
                    not repertoire_snapshots.empty and repertoire_snapshots["selection_counts_available"].all()
                ),
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    print(f"Exported normalized analysis tables to {ANALYSIS_PATH}")

if EXPORT_MARKDOWN_REPORT:
    ANALYSIS_PATH.mkdir(exist_ok=True)
    report_path = ANALYSIS_PATH / "benchmark_report.md"
    write_markdown_report(report_path)
    print(f"Exported self-contained Markdown report to {report_path}")

if not EXPORT_ARTIFACTS and not EXPORT_MARKDOWN_REPORT:
    print("Exports disabled. Set EXPORT_ARTIFACTS or EXPORT_MARKDOWN_REPORT to True in the configuration cell to enable them.")

In [ ]:
import pypowsybl as pp
from pypowsybl_jupyter import network_explorer
from toop_engine_grid_helpers.powsybl.example_grids import (
    create_complex_grid_battery_hvdc_svc_3w_trafo,
)
 
 
net = create_complex_grid_battery_hvdc_svc_3w_trafo()
pp.loadflow.run_ac(net)
component_library = "Convergence"
sld_param = pp.network.SldParameters(use_name=True, component_library = component_library, nodes_infos=True, display_current_feeder_info = True)
nad_parameters=pp.network.NadParameters(edge_info_along_edge=True, substation_description_displayed=True)
network_explorer(net, depth=0, sld_parameters=sld_param, nad_parameters=nad_parameters)